# 23 — Multipartite Entanglement Networks

**Version:** `23_multipartite_entanglement_networks_v1`

Notebook 13 found that symmetric frequency-mode pairs form disconnected two-mode components:

\[
(-1,+1),\ (-2,+2),\ldots,(-n,+n)
\]

Notebook 23 asks:

> **When do isolated symmetric pairs become a network?**

This notebook is a graph model. It does not claim a specific experimental implementation.  
It asks which connectivity patterns could turn many pair resources into multipartite network structure.

In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

VERSION = "23_multipartite_entanglement_networks_v1"
print("running:", VERSION)

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "notebooks").exists():
    ROOT = cwd
else:
    ROOT = cwd

FIGURES_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
CSV_DIR = RESULTS_DIR / "csv"
JSON_DIR = RESULTS_DIR / "json"

for path in [FIGURES_DIR, CSV_DIR, JSON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## 1. Build baseline pair graph

Start from the Notebook 13 structure:

\[
(-n,+n)
\]

Each pair is an edge.

Without extra coupling, each pair is its own connected component.

In [ ]:
n_pairs = 8

def independent_pair_graph(n_pairs=8):
    G = nx.Graph()
    for n in range(1, n_pairs + 1):
        G.add_edge(-n, n, relation="symmetric_pair")
    return G

G_independent = independent_pair_graph(n_pairs)

independent_metrics = {
    "nodes": G_independent.number_of_nodes(),
    "edges": G_independent.number_of_edges(),
    "connected_components": nx.number_connected_components(G_independent),
    "largest_component": max(len(c) for c in nx.connected_components(G_independent)),
}

independent_metrics

In [ ]:
def pair_positions(n_pairs=8):
    pos = {}
    for idx, n in enumerate(range(1, n_pairs + 1)):
        y = n_pairs - idx
        pos[-n] = (-1, y)
        pos[n] = (1, y)
    return pos

pos_pairs = pair_positions(n_pairs)

fig, ax = plt.subplots(figsize=(7, 7))
nx.draw_networkx_nodes(G_independent, pos_pairs, node_size=700, ax=ax)
nx.draw_networkx_edges(G_independent, pos_pairs, width=2, ax=ax)
nx.draw_networkx_labels(G_independent, pos_pairs, labels={node: str(node) for node in G_independent.nodes()}, font_size=10, ax=ax)

ax.set_title("Independent Symmetric Pair Components")
ax.axis("off")

fig.tight_layout()
independent_path = FIGURES_DIR / "23_independent_pair_components.png"
fig.savefig(independent_path, dpi=200)
plt.show()

print("saved:", independent_path)

## 2. Add nearest-neighbor coupling

A minimal way to turn isolated pairs into a network is to add nearest-neighbor edges along the negative and positive frequency ladders:

\[
+n \leftrightarrow +(n+1)
\]

\[
-n \leftrightarrow -(n+1)
\]

This creates a coupled ladder.

In [ ]:
def coupled_pair_ladder_graph(n_pairs=8):
    G = independent_pair_graph(n_pairs)

    for n in range(1, n_pairs):
        G.add_edge(n, n + 1, relation="positive_ladder")
        G.add_edge(-n, -(n + 1), relation="negative_ladder")

    return G

G_ladder = coupled_pair_ladder_graph(n_pairs)

ladder_metrics = {
    "nodes": G_ladder.number_of_nodes(),
    "edges": G_ladder.number_of_edges(),
    "connected_components": nx.number_connected_components(G_ladder),
    "largest_component": max(len(c) for c in nx.connected_components(G_ladder)),
}

ladder_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

edge_styles = {
    "symmetric_pair": {"width": 2.4},
    "positive_ladder": {"width": 1.4},
    "negative_ladder": {"width": 1.4},
}

nx.draw_networkx_nodes(G_ladder, pos_pairs, node_size=700, ax=ax)
nx.draw_networkx_labels(G_ladder, pos_pairs, labels={node: str(node) for node in G_ladder.nodes()}, font_size=10, ax=ax)

for relation, style in edge_styles.items():
    edges = [(u, v) for u, v, d in G_ladder.edges(data=True) if d.get("relation") == relation]
    nx.draw_networkx_edges(G_ladder, pos_pairs, edgelist=edges, width=style["width"], ax=ax)

ax.set_title("Coupled Pair Ladder")
ax.axis("off")

fig.tight_layout()
ladder_path = FIGURES_DIR / "23_coupled_pair_ladder.png"
fig.savefig(ladder_path, dpi=200)
plt.show()

print("saved:", ladder_path)

## 3. Connectivity transition

Starting with isolated pairs, add ladder edges one at a time.

Track how the number of connected components changes.

The transition of interest is:

\[
8\ \text{components} \rightarrow 1\ \text{component}
\]

In [ ]:
def graph_with_k_ladder_edges(n_pairs=8, k=0):
    G = independent_pair_graph(n_pairs)

    ladder_edges = []
    for n in range(1, n_pairs):
        ladder_edges.append((n, n + 1))
        ladder_edges.append((-n, -(n + 1)))

    for edge in ladder_edges[:k]:
        G.add_edge(*edge, relation="added_coupling")

    return G

rows = []
max_extra_edges = 2 * (n_pairs - 1)

for k in range(max_extra_edges + 1):
    Gk = graph_with_k_ladder_edges(n_pairs, k)
    components = list(nx.connected_components(Gk))
    rows.append({
        "added_coupling_edges": k,
        "nodes": Gk.number_of_nodes(),
        "edges": Gk.number_of_edges(),
        "connected_components": nx.number_connected_components(Gk),
        "largest_component": max(len(c) for c in components),
    })

transition = pd.DataFrame(rows)
transition.head(), transition.tail()

In [ ]:
transition_path = CSV_DIR / "23_connectivity_transition.csv"
transition.to_csv(transition_path, index=False)
print("saved:", transition_path)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    transition["added_coupling_edges"],
    transition["connected_components"],
    marker="o",
    label="connected components",
)

ax.plot(
    transition["added_coupling_edges"],
    transition["largest_component"],
    marker="s",
    label="largest component size",
)

ax.set_title("Connectivity Transition from Pairs to Network")
ax.set_xlabel("Added coupling edges")
ax.set_ylabel("Graph metric")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
transition_figure_path = FIGURES_DIR / "23_connectivity_transition.png"
fig.savefig(transition_figure_path, dpi=200)
plt.show()

print("saved:", transition_figure_path)

## 4. Hub-connected model

Another schematic model is a shared pump or shared mediator.

This adds a hub node connected to every frequency mode.

It is not asserted as a literal entanglement graph.  
It represents the idea of shared mediation.

In [ ]:
def hub_connected_graph(n_pairs=8):
    G = independent_pair_graph(n_pairs)
    hub = "pump"
    for n in range(1, n_pairs + 1):
        G.add_edge(hub, -n, relation="hub")
        G.add_edge(hub, n, relation="hub")
    return G

G_hub = hub_connected_graph(n_pairs)

hub_metrics = {
    "nodes": G_hub.number_of_nodes(),
    "edges": G_hub.number_of_edges(),
    "connected_components": nx.number_connected_components(G_hub),
    "largest_component": max(len(c) for c in nx.connected_components(G_hub)),
}

hub_metrics

In [ ]:
pos_hub = {"pump": (0, n_pairs + 1)}
for idx, n in enumerate(range(1, n_pairs + 1)):
    y = n_pairs - idx
    pos_hub[-n] = (-1, y)
    pos_hub[n] = (1, y)

fig, ax = plt.subplots(figsize=(7, 8))

nx.draw_networkx_nodes(G_hub, pos_hub, node_size=700, ax=ax)
nx.draw_networkx_edges(G_hub, pos_hub, width=1.2, ax=ax)
nx.draw_networkx_labels(G_hub, pos_hub, labels={node: str(node) for node in G_hub.nodes()}, font_size=10, ax=ax)

ax.set_title("Hub-Connected Frequency Modes")
ax.axis("off")

fig.tight_layout()
hub_path = FIGURES_DIR / "23_hub_connected_modes.png"
fig.savefig(hub_path, dpi=200)
plt.show()

print("saved:", hub_path)

## 5. Multipartite candidate graph

A connected graph is not automatically useful multipartite entanglement.

But graph connectivity is a necessary structural step.

Here we construct a simple connected candidate graph over the frequency modes.

In [ ]:
def multipartite_candidate_graph(n_pairs=8):
    G = independent_pair_graph(n_pairs)

    # Positive and negative ladders.
    for n in range(1, n_pairs):
        G.add_edge(n, n + 1, relation="ladder")
        G.add_edge(-n, -(n + 1), relation="ladder")

    # Cross links between neighboring pair components.
    for n in range(1, n_pairs):
        G.add_edge(-n, n + 1, relation="cross")
        G.add_edge(n, -(n + 1), relation="cross")

    return G

G_candidate = multipartite_candidate_graph(n_pairs)

candidate_metrics = {
    "nodes": G_candidate.number_of_nodes(),
    "edges": G_candidate.number_of_edges(),
    "connected_components": nx.number_connected_components(G_candidate),
    "largest_component": max(len(c) for c in nx.connected_components(G_candidate)),
}

candidate_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

nx.draw_networkx_nodes(G_candidate, pos_pairs, node_size=700, ax=ax)
nx.draw_networkx_labels(G_candidate, pos_pairs, labels={node: str(node) for node in G_candidate.nodes()}, font_size=10, ax=ax)

relations = sorted(set(d.get("relation") for _, _, d in G_candidate.edges(data=True)))
for relation in relations:
    edges = [(u, v) for u, v, d in G_candidate.edges(data=True) if d.get("relation") == relation]
    width = 2.4 if relation == "symmetric_pair" else 1.2
    nx.draw_networkx_edges(G_candidate, pos_pairs, edgelist=edges, width=width, ax=ax)

ax.set_title("Multipartite Candidate Graph")
ax.axis("off")

fig.tight_layout()
candidate_path = FIGURES_DIR / "23_multipartite_candidate_graph.png"
fig.savefig(candidate_path, dpi=200)
plt.show()

print("saved:", candidate_path)

## 6. Graph metrics table

Compare four schematic graph models:

1. independent pair graph
2. coupled pair ladder
3. hub-connected model
4. multipartite candidate graph

In [ ]:
def metrics_for_graph(name, G, interpretation):
    components = list(nx.connected_components(G))
    return {
        "graph": name,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "connected_components": nx.number_connected_components(G),
        "largest_component": max(len(c) for c in components),
        "interpretation": interpretation,
    }

metrics_table = pd.DataFrame([
    metrics_for_graph("independent_pairs", G_independent, "pairwise only"),
    metrics_for_graph("coupled_pair_ladder", G_ladder, "connected ladder network"),
    metrics_for_graph("hub_connected_modes", G_hub, "shared mediator schematic"),
    metrics_for_graph("multipartite_candidate", G_candidate, "connected candidate network"),
])

metrics_table

In [ ]:
metrics_path = CSV_DIR / "23_graph_metrics.csv"
metrics_table.to_csv(metrics_path, index=False)
print("saved:", metrics_path)

## 7. Summary

Symmetric Kerr pairing creates isolated two-mode components.

Multipartite structure requires additional coupling, routing, or measurement-induced connectivity.

The repo question becomes:

> **Which coupling pattern turns many frequency pairs into one usable quantum network?**

In [ ]:
summary = {
    "notebook": "23_multipartite_entanglement_networks",
    "version": VERSION,
    "question": "When do isolated symmetric pairs become a network?",
    "n_pairs": int(n_pairs),
    "models": [
        "independent_pairs",
        "coupled_pair_ladder",
        "hub_connected_modes",
        "multipartite_candidate"
    ],
    "outputs": [
        "figures/23_independent_pair_components.png",
        "figures/23_coupled_pair_ladder.png",
        "figures/23_connectivity_transition.png",
        "figures/23_hub_connected_modes.png",
        "figures/23_multipartite_candidate_graph.png",
        "results/csv/23_connectivity_transition.csv",
        "results/csv/23_graph_metrics.csv",
        "results/json/23_multipartite_entanglement_networks_summary.json",
        "results/23_multipartite_entanglement_networks_outputs.zip"
    ],
}

summary_path = JSON_DIR / "23_multipartite_entanglement_networks_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))

## 8. Download outputs

Run this cell to package all Notebook 23 outputs.

In Google Colab, it starts a browser download.

In local Jupyter, it prints the zip path.

In [ ]:
zip_path = RESULTS_DIR / "23_multipartite_entanglement_networks_outputs.zip"

files_to_zip = [
    FIGURES_DIR / "23_independent_pair_components.png",
    FIGURES_DIR / "23_coupled_pair_ladder.png",
    FIGURES_DIR / "23_connectivity_transition.png",
    FIGURES_DIR / "23_hub_connected_modes.png",
    FIGURES_DIR / "23_multipartite_candidate_graph.png",
    CSV_DIR / "23_connectivity_transition.csv",
    CSV_DIR / "23_graph_metrics.csv",
    JSON_DIR / "23_multipartite_entanglement_networks_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print("download package ready:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Local Jupyter: download manually from")
    print(zip_path)

In [ ]:
outputs = [
    FIGURES_DIR / "23_independent_pair_components.png",
    FIGURES_DIR / "23_coupled_pair_ladder.png",
    FIGURES_DIR / "23_connectivity_transition.png",
    FIGURES_DIR / "23_hub_connected_modes.png",
    FIGURES_DIR / "23_multipartite_candidate_graph.png",
    CSV_DIR / "23_connectivity_transition.csv",
    CSV_DIR / "23_graph_metrics.csv",
    JSON_DIR / "23_multipartite_entanglement_networks_summary.json",
    RESULTS_DIR / "23_multipartite_entanglement_networks_outputs.zip",
]

for output in outputs:
    print("exists:", output.exists(), "→", output.relative_to(ROOT) if output.exists() else output)

## Takeaway

Notebook 13 found isolated symmetric pair components.

Notebook 23 shows that pair components become a network only after additional connectivity is introduced.

This creates the next research question:

\[
\text{many symmetric pairs}
\rightarrow
\text{coupled graph}
\rightarrow
\text{multipartite candidate network}
\]

The experimentally meaningful version of this question is:

> Which coupling, routing, or measurement structure turns frequency-mode pairs into usable multipartite quantum resources?